# 09 — Train the V3 subject-wise ensemble

Five subject-disjoint folds train compact waveform and STFT CNNs plus an SQI
ExtraTrees model. Every development record receives an out-of-fold prediction.
Calibration and quality-gate thresholds use only these OOF scores; original test
subjects are never opened by this notebook.

In [1]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
RAW_DIR = Path(os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")).expanduser().resolve()
V3_DATA_DIR = Path(os.environ.get("PURE_V3_DATA_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v3")).expanduser().resolve()
V3_MODEL_DIR = Path(os.environ.get("PURE_V3_MODEL_DIR", REPO_ROOT / "models" / "production" / "v3")).expanduser().resolve()

print("Raw:", RAW_DIR)
print("V3 data:", V3_DATA_DIR)
print("V3 models:", V3_MODEL_DIR)

Raw: /mnt/home/gracewahyuni06/pure-pulsesure/data/raw/but-ppg-2.0.0
V3 data: /mnt/home/gracewahyuni06/pure-pulsesure/data/processed/but-ppg-v3
V3 models: /mnt/home/gracewahyuni06/pure-pulsesure/models/production/v3


In [2]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "train_v3_ensemble.py"),
        "--input-dir", str(V3_DATA_DIR),
        "--model-dir", str(V3_MODEL_DIR),
        "--epochs", os.environ.get("PURE_V3_EPOCHS", "50"),
        "--patience", os.environ.get("PURE_V3_PATIENCE", "8"),
    ],
    check=True,
)

Device: cuda; development records: 3248; folds: 5
Fold 1/5: train=2688, validation=560, validation_subjects=[np.str_('102'), np.str_('106'), np.str_('108'), np.str_('110'), np.str_('111'), np.str_('116'), np.str_('123'), np.str_('134'), np.str_('140'), np.str_('141')]
  AUC waveform=0.8912, STFT=0.8632
Fold 2/5: train=2504, validation=744, validation_subjects=[np.str_('112'), np.str_('122'), np.str_('124'), np.str_('125'), np.str_('132'), np.str_('136'), np.str_('145')]
  AUC waveform=0.8558, STFT=0.8469
Fold 3/5: train=2596, validation=652, validation_subjects=[np.str_('107'), np.str_('115'), np.str_('118'), np.str_('128'), np.str_('138'), np.str_('146'), np.str_('147'), np.str_('148')]
  AUC waveform=0.7887, STFT=0.7712
Fold 4/5: train=2558, validation=690, validation_subjects=[np.str_('114'), np.str_('117'), np.str_('120'), np.str_('121'), np.str_('127'), np.str_('135'), np.str_('144')]
  AUC waveform=0.8906, STFT=0.8681
Fold 5/5: train=2646, validation=602, validation_subjects=[np.

CompletedProcess(args=['/mnt/home/gracewahyuni06/pure-pulsesure/.venv/bin/python', '/mnt/home/gracewahyuni06/pure-pulsesure/scripts/train_v3_ensemble.py', '--input-dir', '/mnt/home/gracewahyuni06/pure-pulsesure/data/processed/but-ppg-v3', '--model-dir', '/mnt/home/gracewahyuni06/pure-pulsesure/models/production/v3', '--epochs', '50', '--patience', '8'], returncode=0)

In [3]:
import json
import pandas as pd

report = json.loads((V3_MODEL_DIR / "oof_metrics.json").read_text())
display(pd.Series(report["oof_binary"]).to_frame("OOF ensemble"))
display(pd.Series(report["oof_gate"]).to_frame("OOF quality gate"))
display(pd.DataFrame(report["branch_oof_metrics"]).T[["accuracy", "balanced_accuracy", "f1_good", "roc_auc", "false_acceptance_rate"]])

,OOF ensemble
threshold,0.235
accuracy,0.86484
balanced_accuracy,0.809647
precision_good,0.652414
recall_good_sensitivity,0.716667
specificity_poor,0.902628
f1_good,0.683032
false_acceptance_rate,0.097372
false_rejection_rate,0.283333
roc_auc,0.890224


,OOF quality gate
lower_threshold,0.073191
upper_threshold,0.410678
false_acceptance_rate,0.049845
false_rejection_rate,0.050000
good_acceptance_rate,0.589394
poor_rejection_rate,0.527048
reliable_coverage,0.159483
caution_rate,0.410406
unreliable_rate,0.430111


,accuracy,balanced_accuracy,f1_good,roc_auc,false_acceptance_rate
waveform,0.850985,0.777249,0.640416,0.856299,0.098532
stft,0.845751,0.764371,0.623025,0.84207,0.098532
sqi,0.857759,0.792787,0.66129,0.878141,0.097759
raw_equal_weight_ensemble,0.867303,0.810628,0.686545,0.890224,0.093895
